 Live survey response analysis

In [1]:
import json
from urllib.request import Request, urlopen

import pandas as pd
from IPython.display import Markdown, display

from survey_responce import rank_llms_by_category, summarize_participants


In [2]:
config_path = "live-analisys-config.json"
survey_path = "../eval/output/survey.json"

with open(config_path, encoding="utf-8") as config_file:
	config = json.load(config_file)


def fetch_api(path):
	request = Request(
		f"{config['base_url'].rstrip('/')}{path}",
		headers={"Authorization": f"Bearer {config['admin_token']}"},
	)
	with urlopen(request, timeout=30) as api_response:
		return json.load(api_response)


responses = fetch_api("/api/responses")
participants = fetch_api("/api/participants/")


 Responses by participant

In [3]:
participant_stats = pd.DataFrame(
	summarize_participants(responses, participants, survey_path),
	columns=[
		"participant", "group", "status", "last_page", "saved_at",
		"completed_cases", "incomplete_cases", "complete_rankings",
	],
).astype({"group": "Int64", "last_page": "Int64"})
status_counts = participant_stats["status"].value_counts()
display(Markdown(
	f"**{len(participant_stats)}** participants with responses: "
	f"**{status_counts.get('submitted', 0)}** submitted, **{status_counts.get('draft', 0)}** drafts; "
	f"**{participant_stats['completed_cases'].sum()}** completed cases and "
	f"**{participant_stats['complete_rankings'].sum()}** complete rankings in total."
))
participant_stats.sort_values(["status", "participant"], ascending=[False, True], ignore_index=True)


**1** participants with responses: **0** submitted, **1** drafts; **2** completed cases and **12** complete rankings in total.

,participant,group,status,last_page,saved_at,completed_cases,incomplete_cases,complete_rankings
0,"Preview participant {""group"": 2}",2,draft,2,2026-10-02T12:45:45.846731,2,0,12


 Plackett-Luce rankings by category

In [4]:
# Drafts are autosaved on every change, so rows without both best and worst are skipped.
rankings_by_category = rank_llms_by_category(responses, survey_path, skip_incomplete=True)
for category, rankings in rankings_by_category.items():
	display(Markdown(f"### {category}"))
	display(pd.DataFrame(rankings).set_index("rank"))
if not rankings_by_category:
	display(Markdown("No complete rankings yet."))


### Tačnost navedenih informacija

,model,score
rank,,
1,Qwen3-14B-t7-sft,0.999925
2,gpt-5.2,0.000037
3,nvidia/Llama-3.3-70B-Instruct-FP8,0.000037
4,Qwen/Qwen3-32B,0.000000


### Usklađenost sa gradivom

,model,score
rank,,
1,nvidia/Llama-3.3-70B-Instruct-FP8,0.381966
2,Qwen/Qwen3-32B,0.381966
3,gpt-5.2,0.236068
4,Qwen3-14B-t7-sft,0.000000


### Primerenost nivou znanja i mogućnostima učenika

...

,model,score
rank,,
1,nvidia/Llama-3.3-70B-Instruct-FP8,0.381966
2,Qwen/Qwen3-32B,0.381966
3,gpt-5.2,0.236068
4,Qwen3-14B-t7-sft,0.000000


### Jasnoća i prirodnost srpskog jezika

,model,score
rank,,
1,gpt-5.2,0.99995
2,nvidia/Llama-3.3-70B-Instruct-FP8,0.00005
3,Qwen/Qwen3-32B,0.00000
4,Qwen3-14B-t7-sft,0.00000


### Koji biste odgovor najradije iskoristili u praksi, uz moguće dorade?

,model,score
rank,,
1,Qwen3-14B-t7-sft,0.9999
2,gpt-5.2,0.0001
3,Qwen/Qwen3-32B,0.0000
4,nvidia/Llama-3.3-70B-Instruct-FP8,0.0000
